# Offline sweep result visualizer
Reads HDF5 scalars + matplotlib-saved images for a given task ID.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image

TASK_ID = "PASTE_TASK_ID_HERE"
LOG_DIR = Path("/scratch/wlp9800/offline_logs")

h5_path = LOG_DIR / f"metrics_{TASK_ID}.h5"
img_dir = LOG_DIR / f"images_{TASK_ID}"
print(f"h5: {h5_path}  (exists: {h5_path.exists()})")
print(f"img dir: {img_dir}  (exists: {img_dir.exists()})")

## List all logged scalar series

In [ ]:
with h5py.File(h5_path, "r") as f:
    titles = sorted(k for k in f.keys() if not k.endswith("_iterations"))
for t in titles:
    print(t)

## Helper: plot one scalar

In [ ]:
def plot_scalar(title: str):
    with h5py.File(h5_path, "r") as f:
        iters = f[f"{title}_iterations"][:]
        vals = f[title][:]
    mask = (iters >= 0) & ~np.isnan(vals)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(iters[mask], vals[mask], linewidth=1)
    ax.set_xlabel("iteration"); ax.set_ylabel("value"); ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.show()

# Example:
# plot_scalar("train/level0/loss")

## Plot specific scalars
Add one cell per series you want to look at.

In [ ]:
plot_scalar("train/level0/loss")

In [ ]:
plot_scalar("train/level0/kl")

In [ ]:
plot_scalar("train/kl_regularizer_beta/meta1_beta/0")

In [ ]:
plot_scalar("train/learning_rate/meta1_sgd1_lr/0")

In [ ]:
plot_scalar("train/weight_decay/meta1_sgd1_wd/0")

In [ ]:
plot_scalar("sample/disentanglement/level0/tc")

## Plot all scalars in a grid

In [ ]:
with h5py.File(h5_path, "r") as f:
    titles = sorted(k for k in f.keys() if not k.endswith("_iterations"))
    series = {}
    for t in titles:
        iters = f[f"{t}_iterations"][:]
        vals = f[t][:]
        mask = (iters >= 0) & ~np.isnan(vals)
        series[t] = (iters[mask], vals[mask])

ncols = 3
nrows = (len(series) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes
for ax, (title, (its, vs)) in zip(axes, series.items()):
    ax.plot(its, vs, linewidth=1)
    ax.set_title(title, fontsize=8); ax.grid(True, alpha=0.3)
for ax in axes[len(series):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## Image gallery
Shows every PNG saved by the HDF5 logger's internal MatplotlibLogger.

In [ ]:
images = sorted(img_dir.rglob("*.png")) if img_dir.exists() else []
print(f"{len(images)} images found")
for p in images[:20]:
    print(f"\n=== {p.relative_to(img_dir)} ===")
    plt.figure(figsize=(6, 6))
    plt.imshow(np.asarray(Image.open(p)))
    plt.axis("off"); plt.show()